# Mamografi Sınıflandırma — Colab'da Eğitim

Bu defter her şeyi baştan sona yapar: kodu indirir, MIAS veri setini indirir, modeli eğitir, test eder ve uygulamayı açar.

**Başlamadan önce:** Menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
Sonra **Çalışma zamanı → Tümünü çalıştır** deyin.

> ⚠️ Araştırma/eğitim amaçlıdır, tıbbi teşhis yerine geçmez.

## 1. Kodu indir ve kur

In [ ]:
import os
if not os.path.exists("/content/Deneme"):
    !git clone -b claude/breast-cancer-classification-app-pwhehx https://github.com/makifykaya/Deneme.git /content/Deneme
%cd /content/Deneme
!git pull -q
!pip install -q pydicom kagglehub

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK — Çalışma zamanı türünü T4 GPU yapın!")

## 2. MIAS veri setini indir
322 mamografi görüntüsü: normal / iyi huylu / kötü huylu. Önce Kaggle'dan denenir, olmazsa orijinal kaynaktan indirilir.

In [ ]:
import glob, shutil, tarfile, zipfile, subprocess
from pathlib import Path

MIAS = Path("/content/mias")
MIAS.mkdir(exist_ok=True)

def extract_archives(root):
    for p in list(Path(root).rglob("*")):
        if p.suffix in {".gz", ".tar", ".tgz"} and tarfile.is_tarfile(p):
            with tarfile.open(p) as t: t.extractall(p.parent)
        elif p.suffix == ".zip":
            with zipfile.ZipFile(p) as z: z.extractall(p.parent)

def have_mias():
    return len(list(MIAS.rglob("mdb*.pgm"))) > 300

if not have_mias():
    try:
        import kagglehub
        src = kagglehub.dataset_download("kmader/mias-mammography")
        shutil.copytree(src, MIAS, dirs_exist_ok=True)
        extract_archives(MIAS)
    except Exception as e:
        print("Kaggle indirmesi başarısız:", e)

if not have_mias():
    subprocess.run(["wget", "-q", "-O", str(MIAS / "all-mias.tar.gz"),
                    "http://peipa.essex.ac.uk/pub/mias/all-mias.tar.gz"], check=False)
    extract_archives(MIAS)

print("Görüntü sayısı:", len(list(MIAS.rglob("mdb*.pgm"))))
print("Info dosyası:", [str(p) for p in MIAS.rglob("*") if p.name.lower() in {"info.txt", "info"}])

## 3. Eğitim / doğrulama / test olarak böl

In [ ]:
!rm -rf data/processed
!python -m src.prepare_data --mias /content/mias --dst data/processed

## 4. Modeli eğit
T4 GPU'da yaklaşık 10–20 dakika sürer. Her epoch'ta doğrulama F1 skoru yazdırılır; en iyi model `models/best_model.pt` olarak kaydedilir.

In [ ]:
!python -m src.train --data data/processed --arch efficientnet_b0 --epochs 40 --batch-size 16 --img-size 384 --patience 10 --workers 2

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open("models/best_model.history.json"))
ep = [r["epoch"] for r in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [r["train_loss"] for r in h], label="eğitim"); ax[0].plot(ep, [r["val_loss"] for r in h], label="doğrulama")
ax[0].set_title("Kayıp"); ax[0].legend()
ax[1].plot(ep, [r["train_f1"] for r in h], label="eğitim"); ax[1].plot(ep, [r["val_f1"] for r in h], label="doğrulama")
ax[1].set_title("Makro F1"); ax[1].legend()
plt.show()

## 5. Test kümesinde değerlendir

In [ ]:
!python -m src.evaluate --data data/processed/test --checkpoint models/best_model.pt --save-dir outputs
from IPython.display import Image as IPImage, display
display(IPImage("outputs/confusion_matrix.png"))

## 6. Kendi görüntünüzle deneyin
Aşağıdaki hücre bir dosya yükleme düğmesi açar.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
from src.config import CLASS_LABELS_TR
from src.data import load_image
from src.predict import Predictor, overlay_heatmap

pred = Predictor("models/best_model.pt")
for name, content in files.upload().items():
    open(name, "wb").write(content)
    img = load_image(name)
    probs = pred.predict(img)
    best = max(probs, key=probs.get)
    cam = pred.gradcam(img, pred.classes.index(best))
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img); ax[0].set_title("Görüntü"); ax[0].axis("off")
    ax[1].imshow(overlay_heatmap(img, cam)); ax[1].set_title("Grad-CAM"); ax[1].axis("off")
    plt.suptitle(f"{CLASS_LABELS_TR[best]} — %{probs[best]*100:.1f}")
    plt.show()
    for c in sorted(probs, key=probs.get, reverse=True):
        print(f"{CLASS_LABELS_TR[c]:<25} %{probs[c]*100:.1f}")

## 7. (İsteğe bağlı) Web uygulamasını Colab'da aç
Streamlit uygulaması başlatılır ve geçici bir genel bağlantı (`https://....trycloudflare.com`) verilir. Colab oturumu açık kaldığı sürece çalışır.

In [ ]:
!pip install -q streamlit
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import subprocess, re, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("Uygulama bağlantısı:", m.group(0))
        break

## 8. Modeli bilgisayarınıza indirin
Colab oturumu kapanınca dosyalar silinir. Modeli indirip projenin `models/` klasörüne koyarsanız uygulamayı kendi bilgisayarınızda da çalıştırabilirsiniz.

In [ ]:
from google.colab import files
files.download("models/best_model.pt")